# test
> Run unit tests on notebooks in parallel
- order: 12

In [ ]:
#| default_exp test

In [ ]:
#| export
import time,os,sys,io,traceback,contextlib,inspect,signal,asyncio,threading
from fastcore.basics import *
from fastcore.imports import *
from fastcore.foundation import *
from fastcore.parallel import *
from fastcore.script import *
from fastcore.meta import delegates

from nbdev.config import *
from nbdev.doclinks import *
from nbdev.process import NBProcessor, nb_lang
from nbdev.frontmatter import nb_frontmatter

from fastcore.nbio import *
from execnb.shell import *

When a test run hangs, ctrl-c should say what each notebook was executing. Each worker installs a SIGINT handler that prints the in-flight notebook's stack — the sync frames plus every pending asyncio task, since a hang inside an awaited coroutine lives on the task, not the sync stack — as a single block (one `os.write`, so parallel workers' dumps don't interleave) and then exits immediately, rather than letting the interrupt surface inside the cell where the shell would catch it and carry on. An idle worker exits silently.

In [ ]:
#| export
_cur_nb = [None]
_cur_shell = [None]

def _await_chain(t):
    "One frame per coroutine in task `t`'s await chain, deepest last: where a suspended hang actually sits"
    co = t.get_coro()
    while co is not None:
        f = getattr(co, 'cr_frame', None) or getattr(co, 'ag_frame', None) or getattr(co, 'gi_frame', None)
        if f is not None: yield f, f.f_lineno
        co = getattr(co, 'cr_await', None) or getattr(co, 'ag_await', None) or getattr(co, 'gi_yieldfrom', None)

def _int_handler(signum, frame):
    "Dump the running notebook's stack and exit; installed on SIGINT by `test_nb`"
    if _cur_nb[0] is not None:
        buf = io.StringIO()
        traceback.print_stack(frame, file=buf)
        k = _cur_shell[0]
        if k is not None:
            f = sys._current_frames().get(k._thread.ident)
            if f is not None:
                print('\n--- shell loop thread ---', file=buf)
                traceback.print_stack(f, file=buf)
            with contextlib.suppress(RuntimeError):
                for t in asyncio.all_tasks(k.loop):
                    print(f'\n{t}', file=buf)
                    buf.writelines(traceback.StackSummary.extract(_await_chain(t)).format())
        os.write(2, f'\n=== nbdev-test interrupted: {_cur_nb[0]} ===\n{buf.getvalue()}'.encode())
    os._exit(130)

In [ ]:
#| export
def test_nb(
    fn,  # file name of notebook to test
    skip_flags=None,  # list of flags marking cells to skip
    force_flags=None,  # list of flags marking cells to always run
    do_print=False,  # print completion?
    showerr=True,  # print errors to stderr?
    basepath=None,  # path to add to sys.path
    verbose=False,  # stream stdout/stderr from cells to console?
    save=False,  # write outputs back to notebook on success?
    profile:bool=None, # load the IPython profile, as `ipykernel` does? (default: `exec_profile` config key)
    cell_timeout:int=600, # seconds before each cell times out (None: no limit)
    cell_timing_min:float=None # print cells slower than this many seconds (None: no timing output)
):
    "Execute tests in notebook in `fn` except those with `skip_flags`"
    if not IN_NOTEBOOK and threading.current_thread() is threading.main_thread(): signal.signal(signal.SIGINT, _int_handler)
    fn = Path(fn)
    _cur_nb[0] = fn
    if basepath: sys.path.insert(0, str(basepath))
    prev_test = os.environ.get('IN_TEST')
    if not IN_NOTEBOOK: os.environ['IN_TEST'] = '1'
    try:
        flags=set(L(skip_flags)) - set(L(force_flags))
        nb = NBProcessor(fn, rm_directives=False, process=True).nb
        fm = nb_frontmatter(nb)
        if str2bool(fm.get('skip_exec', False)) or nb_lang(nb) != 'python': return True, 0

        dflt = fm_default_eval(fm)
        def _no_eval(cell):
            if cell.cell_type != 'code': return True
            if not does_cell_eval(cell, dflt): return True
            return flags & (getattr(cell, 'directives_', {}) or {}).keys()

        def _postproc(cell):
            elapsed = cell.metadata['execution']['total']
            if cell_timing_min is not None and elapsed > cell_timing_min: print(f'{fn.name}:{cell.id}: {elapsed:.3f}s')

        start = time.time()
        if profile is None: profile = bool(get_config(fn.parent).exec_profile)
        k = CaptureShell(fn, profile=profile)
        _cur_shell[0] = k
        if do_print: print(f'Starting {fn}')
        try:
            with working_directory(fn.parent):
                k.run_all(nb, exc_stop=True, preproc=_no_eval, postproc=_postproc, verbose=verbose, cell_timeout=cell_timeout)
                if save: write_nb(nb, fn)
                res = True
        except: 
            if showerr: sys.stderr.write(k.prettytb(fname=fn)+'\n')
            res=False
        if k.leaks and showerr: sys.stderr.write(f'{fn}: {len(k.leaks)} leaked task(s) survived cancellation\n')
        if do_print: print(f'- Completed {fn}')
        return res,time.time()-start
    finally:
        _cur_nb[0] = _cur_shell[0] = None
        if prev_test is None: os.environ.pop('IN_TEST', None)
        else: os.environ['IN_TEST'] = prev_test


`test_nb` is sync: every cell runs on the `CaptureShell`'s own loop thread, and the caller simply blocks on each result. `parallel` workers call it directly, and because the worker's main thread stays free while cells run, the SIGINT handler above can dump the shell thread's stack and its pending tasks in the middle of a hang. `cell_timeout` bounds each cell even when it blocks the loop in sync code -- the timed-out notebook fails with a `TimeoutError` naming the cell and the stuck tasks, instead of hanging the run.


`test_nb` can test a notebook, and skip over certain flags. A notebook whose frontmatter sets `skip_exec: true` (e.g. as a `- skip_exec: true` list item in its title cell) is skipped entirely and reported as passing; use it for notebooks that can't run under test at all, such as those needing credentials or live services:

In [ ]:
_nb = Path('../../tests/directives.ipynb')
success,duration = test_nb(_nb, skip_flags=['notest'])
assert success

In that notebook the cell flagged *notest* raises an exception, which will be returned as a `bool`:

In [ ]:
_nb = Path('../../tests/directives.ipynb')
success,duration = test_nb(_nb, showerr=False)
assert not success

In [ ]:
import tempfile
from fastcore.xtras import modified_env

Pass `cell_timing_min` to identify slow cells without flooding ordinary test output. Each cell over the threshold prints its notebook name, stable cell id, and elapsed seconds; `--cell-timing-min=0.1` is a useful first pass.

`test_nb` loads the IPython profile by default, like `ipykernel` does, so notebooks are tested the way their author's kernel ran them (startup files, extensions, shell config). Set `exec_profile = false` under `[tool.nbdev]`, or pass `profile=False`, to run without it:

In [ ]:
with tempfile.TemporaryDirectory() as td:
    td = Path(td)
    (td/'profile_default'/'startup').mkdir(parents=True)
    (td/'profile_default'/'startup'/'00.py').write_text('prof_x = 7\n')
    nbp = td/'prof.ipynb'
    write_nb(new_nb([mk_cell('assert prof_x==7')]), nbp)
    with modified_env(IPYTHONDIR=str(td)):
        assert     (test_nb(nbp, showerr=False))[0]
        assert not (test_nb(nbp, showerr=False, profile=False))[0]

In [ ]:
#| export
def _keep_file(
    p:Path, # filename for which to check for `indicator_fname`
    ignore_fname:str # filename that will result in siblings being ignored
) -> bool:
    "Returns False if `indicator_fname` is a sibling to `fname` else True"
    if p.exists(): return not bool(p.parent.ls().attrgot('name').filter(lambda x: x == ignore_fname))
    else: True

Sometimes you may wish to override one or more of the skip_flags, in which case you can use the argument `force_flags` which will remove the appropriate tag(s) from `skip_flags`.  This is useful because `skip_flags` are meant to be set in the `tst_flags` field of `[tool.nbdev]` in `pyproject.toml`, whereas `force_flags` are usually passed in by the user.

In [ ]:
#| export
@call_parse
@delegates(nbglob_cli)
def nbdev_test(
    path:str=None,  # A notebook name or glob to test
    flags:str='',  # Space separated list of test flags to run that are normally ignored
    n_workers:int=None,  # Number of workers
    timing:bool=False,  # Time each notebook to see which are slow
    do_print:bool=False, # Print start and end of each notebook
    pause:float=0.01,  # Pause time (in seconds) between notebooks to avoid race conditions
    ignore_fname:str='.notest', # Filename that will result in siblings being ignored
    verbose:bool=False, # Print stdout/stderr from notebook cells?
    save:bool=False, # Write outputs back to notebooks on success?
    cell_timeout:int=600, # Seconds before each cell times out (0: no limit)
    cell_timing_min:float=None, # Print cells slower than this many seconds (None: no timing output)
    **kwargs
):
    "Test in parallel notebooks matching `path`, passing along `flags`"
    cfg = get_config(Path(path).resolve() if path else None)
    skip_flags = cfg.tst_flags
    if isinstance(skip_flags, str): skip_flags = skip_flags.split()
    force_flags = flags.split()
    files = nbglob(path, as_path=True, **kwargs)
    files = [f.absolute() for f in sorted(files) if _keep_file(f, ignore_fname)]
    if len(files)==0: return print('No files were eligible for testing')

    if n_workers is None: n_workers = 0 if len(files)==1 else min(num_cpus(), 8)
    if IN_NOTEBOOK: kw = {'method':'spawn'} if os.name=='nt' or sys.platform=='darwin' else {'method':'forkserver'}
    else: kw = {'method':'spawn'} if sys.platform=='darwin' else {}
    wd_pth = cfg.nbs_path
    with working_directory(wd_pth if (wd_pth and wd_pth.exists()) else os.getcwd()):
        try:
            results = parallel(test_nb, files, skip_flags=skip_flags, force_flags=force_flags, n_workers=n_workers,
                               basepath=cfg.config_path, pause=pause, do_print=do_print, verbose=verbose, save=save,
                               cell_timeout=cell_timeout or None, cell_timing_min=cell_timing_min, **kw)
        except KeyboardInterrupt:
            sys.stderr.write('\nnbdev-test interrupted; in-flight notebook stacks shown above\n')
            sys.exit(130)
    passed,times = zip(*results)
    if all(passed): print("Success.")
    else: 
        _fence = '='*50
        failed = '\n\t'.join(f.name for p,f in zip(passed,files) if not p)
        sys.stderr.write(f"\nnbdev Tests Failed On The Following Notebooks:\n{_fence}\n\t{failed}\n")
        sys.exit(1)
    if timing:
        for i,t in sorted(enumerate(times), key=lambda o:o[1], reverse=True): print(f"{files[i].name}: {int(t)} secs")

In [ ]:
#| eval:false
nbdev_test(n_workers=0)

Success.


You can even run `nbdev-test` in non nbdev projects, for example, you can test an individual notebook like so:

```
nbdev-test --path ../../tests/minimal.ipynb --do_print
```

Or you can test an entire directory of notebooks filtered for only those that match a regular expression:

```
nbdev-test --path ../../tests --file_re '.*test.ipynb' --do_print
```

## Eval

`test_nb` decides which cells run through the `eval` cascade (`fastcore.nbio.does_cell_eval`): a cell's own `#| eval:` directive wins; otherwise the notebook-level `eval` directive — in frontmatter, or the notebook's `metadata.nbdev` mapping — sets the default; with neither, cells run. `#| eval: false` therefore skips one cell, as it always has, while a notebook-level `eval: false` flips the whole notebook to opt-in: only cells marked `#| eval: true` run, which suits a slow or service-dependent notebook where just a few cells are worth testing. Unlike `skip_exec: true`, which skips a notebook unconditionally, marked cells still run — here the unmarked cell would raise if executed, so the passing test is the proof it was skipped:

In [ ]:
with tempfile.TemporaryDirectory() as td:
    cells = [mk_cell('---\neval: false\n---', 'raw'), mk_cell('raise Exception("unmarked: must not run")'),
        mk_cell('#| eval: true\nx = 1')]
    fn = Path(td)/'optin.ipynb'
    write_nb(new_nb(cells), fn)
    success,_ = test_nb(fn)
assert success

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()